# Distribution-free UQ for weak lensing mass mapping
*Notebook reproducing the results from H. Leterme, J. Fadili, and J.-L. Starck, “Distribution-free uncertainty quantification for inverse problems: Application to weak lensing mass mapping,” A&A, vol. 694, p. A267, Feb. 2025.*

**⚠️ WARNING:** This notebook is a work in progress. For a stable version with compatible dependencies, please use `git checkout distribution_free_uq` and open `wlmmuq.ipynb`.

In [ ]:
%config InlineBackend.figure_format = 'svg'
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle
import random

import numpy as np
import torch
from scipy import stats
import matplotlib.pyplot as plt

import wlmmuq as wl
import wlmmuq.datasets.kappatng as wlktng
import wlmmuq.datasets.cosmos as wlcosmos
import wlmmuq.datasets.torch as wlds
import wlmmuq.models.cqr as wlcqr
import wlmmuq.models.rcps as wlrcps
import wlmmuq.utils as wlutils

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
path_to_test_dataset = wl.PATH_TO_TEST_DATASET

In [ ]:
nimgs_test = 32
nimgs_calib = 256
batch_size = 8
imgsize = 384
imgsize_imshow = 304

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Load noise standard deviation and noise

Diagonal elements of $\Sigma$ and masked pixels.

The noise standard deviation has been obtained by binning the COSMOS shape galaxy catalog (https://www.aanda.org/articles/aa/abs/2010/08/aa13577-09/aa13577-09.html) at a resolution of $0.29$ arcmin per pixel, to match the $\kappa$-TNG dataset of simulated convergence maps (https://academic.oup.com/mnras/article/502/4/5593/6133466).

The noise standard deviation is given, for each pixel $k$, by 
\begin{equation}
    \sigma_k = \frac{\sigma_{\text{ell}}}{\sqrt{N_k}},
\end{equation}
where $\sigma_{\text{ell}}$ denotes the intrinsic shape dispersion of galaxies, and $N_k$ denotes the number of measured galaxies.

In [ ]:
cat_cosmos_bright, _ = wlcosmos.cosmos_catalog()
cat_cosmos_bright = wlcosmos.filter_by_redshifts(cat_cosmos_bright, wlktng.MAX_Z)
cosmos_data = wlcosmos.get_data_from_cosmos(cat_cosmos_bright, imgsize, wlktng.RESOLUTION)

extent = cosmos_data.extent # For imshow
shapedisp = cosmos_data.shapedisp # Intrinsic shape dispersion
std_noise = cosmos_data.std_noise # Noise standard deviation
mask = cosmos_data.mask # Mask

In [ ]:
ra, dec = np.array(wlcosmos.COSMOS_VERTICES).T # Survey boundaries

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(121)
wlutils.skyshow(
    std_noise, imgsize=imgsize_imshow,
    boundaries=(ra, dec), extent=extent
)
plt.title("Noise standard deviation per pixel")
plt.subplot(122)
wlutils.skyshow(
    mask, imgsize=imgsize_imshow,
    boundaries=(ra, dec), extent=extent
)
plt.title("Mask")
plt.show()

## Mass mapping before calibration

Mass mapping on the simulated kappaTNG dataset

K. Osato, J. Liu, and Z. Haiman, “κTNG: effect of baryonic processes on weak lensing with IllustrisTNG simulations,” Monthly Notices of the Royal Astronomical Society, vol. 502, no. 4, pp. 5593–5602, Apr. 2021.

In [ ]:
test_dataloader_massmapping = wlds.HDF5DatasetMassMapping(
    hdf5_filepath=path_to_test_dataset,
    nimgs=nimgs_test, batch_size=batch_size,
    output_shape=imgsize,
    std_noise=std_noise, mask=mask,
    shuffle=False, newaxis=True
).to_dataloader()

### Kaiser-Squires estimator

In [ ]:
fwhm_gaussianfilter_arcmin = 2.4 # as in Starck et al. (2021)
std_gaussianfilter_arcmin = fwhm_gaussianfilter_arcmin / (2 * np.sqrt(2 * np.log(2)))
std_gaussianfilter = std_gaussianfilter_arcmin / resolution # pixels

In [ ]:
_, kappa_ks_lo, kappa_ks_hi, _ = wlutils.ksfilter(
    gamma1_noisy, gamma2_noisy, std_noise=std_noise,
    confidence=confidence, std_gaussianfilter=std_gaussianfilter
)
kappa_ks = (kappa_ks_lo + kappa_ks_hi) / 2 # point estimate
res_ks = (kappa_ks_hi - kappa_ks_lo) / 2 # residual

In [ ]:
%xdel kappa_ks_lo
%xdel kappa_ks_hi

In [ ]:
skyshow_pred_bounds(
    kappa_ks[idx], res_ks[idx], kappa[idx]
)

In [ ]:
%xdel gamma1_noisy
%xdel gamma2_noisy

### Iterative Wiener algorithm

Get point estimate. From the root directory, run:

In [ ]:
print(f"python massmapping.py wiener {idx_lp} wiener.pred --ninpimgs {ninpimgs} --seed 42 -v")

Get empirical standard deviation by propagating noise realizations

In [ ]:
Nrea = 25 # number of noise realizations for Monte-Carlo UQ
batch_size = kappa.shape[0] // Nrea # to avoid memory overload

From the root directory, run:

In [ ]:
print(
    f"python massmapping.py wiener {idx_lp} wiener.uq",
    f"--ninpimgs {ninpimgs} -b {batch_size} --uq --nsamples {Nrea} --seed 42 -v"
)

Load results

In [ ]:
pickle_dir = os.path.expanduser(wl.CONFIG_DATA['pickle_dir'])

fn = os.path.join(pickle_dir, 'wiener.pred')
with open(fn, 'rb') as f:
    _, kappa_wiener = pickle.load(f)
kappa_wiener = kappa_wiener[ktng.list_of_idx] # useful when shuffle=True

fn = os.path.join(pickle_dir, 'wiener.uq')
with open(fn, 'rb') as f:
    _, std_wiener = pickle.load(f)
std_wiener = std_wiener[ktng.list_of_idx] # useful when shuffle=True
res_wiener = confidence * std_wiener # residual

In [ ]:
%xdel std_wiener

In [ ]:
skyshow_pred_bounds(kappa_wiener[idx], res_wiener[idx], kappa[idx])

### MCALens

In [ ]:
Nsigma = 4 # detection threshold for selecting the set of active wavelet coefficients

Get point estimate. From the root directory, run:

In [ ]:
print(
    f"python massmapping.py mcalens {idx_lp} mcalens.pred",
    f"--Nsigma {Nsigma} --ninpimgs {ninpimgs} --seed 42 -v"
)

Get empirical standard deviation by propagating noise realizations.
From the root directory, run:

In [ ]:
print(
    f"python massmapping.py mcalens {idx_lp} mcalens.uq --Nsigma {Nsigma}",
    f"--ninpimgs {ninpimgs} -b {batch_size} --uq --nsamples {Nrea} --seed 42 -v"
)

Load results

In [ ]:
fn = os.path.join(pickle_dir, 'mcalens.pred')
with open(fn, 'rb') as f:
    _, kappa_mcalens = pickle.load(f)
kappa_mcalens = kappa_mcalens[ktng.list_of_idx] # useful when shuffle=True

fn = os.path.join(pickle_dir, 'mcalens.uq')
with open(fn, 'rb') as f:
    _, std_mcalens = pickle.load(f)
std_mcalens = std_mcalens[ktng.list_of_idx] # useful when shuffle=True
res_mcalens = confidence * std_mcalens # residual

In [ ]:
%xdel std_mcalens

In [ ]:
skyshow_pred_bounds(kappa_mcalens[idx], res_mcalens[idx], kappa[idx])

## Results before calibration

### Split test and calibration sets

In [ ]:
nimgs_calib = 100
nimgs = kappa.shape[0]
nimgs_test = nimgs - nimgs_calib
print(f"Size of the calibration set = {nimgs_calib}")
print(f"Size of the test set = {nimgs_test}")

In [ ]:
[
    kappa_calib,
    kappa_ks_calib, kappa_wiener_calib, kappa_mcalens_calib,
    res_ks_calib, res_wiener_calib, res_mcalens_calib,
], [
    kappa_test,
    kappa_ks_test, kappa_wiener_test, kappa_mcalens_test,
    res_ks_test, res_wiener_test, res_mcalens_test,
] = \
    wlutils.split_test_calib([
        kappa,
        kappa_ks, kappa_wiener, kappa_mcalens,
        res_ks, res_wiener, res_mcalens,
    ], nimgs_calib, calib_first=False) # first the test set; then the calibration set

print(f"Size of the calibration set = {kappa_calib.shape[0]}")
print(f"Size of the test set = {kappa_test.shape[0]}")

In [ ]:
%xdel kappa
%xdel kappa_ks
%xdel kappa_wiener
%xdel kappa_mcalens
%xdel res_ks
%xdel res_wiener
%xdel res_mcalens

### Accuracy of point estimates

In [ ]:
mse_ks_test = wlutils.normalized_mse(kappa_ks_test, kappa_test, mask=mask)
mse_wiener_test = wlutils.normalized_mse(kappa_wiener_test, kappa_test, mask=mask)
mse_mcalens_test = wlutils.normalized_mse(kappa_mcalens_test, kappa_test, mask=mask)

In [ ]:
def plot_means_errs(
        list_of_means, list_of_stds, xticklabels=None, sec_xticklabels=None,
        xlabel=None, ylabel=None, drawtarget=True, drawbounds=True,
        y_lower=None, y_upper=None, logscale=False, ymin=None, ymax=None, loclegend=None,
        figsize=(6, 3)
):
    """
    Plot means with error bars representing standard deviations
    
    """
    nseries = len(list_of_means)
    assert len(list_of_stds) == nseries
    if xticklabels is not None:
        nvals = len(xticklabels)
    else:
        nvals = 1
    offset = 0.2  # Adjust the offset as needed
    labels = ["KS", "Wiener", "MCALens"]

    _, ax = plt.subplots(figsize=figsize)
    for i, (means, stds, label) in enumerate(zip(list_of_means, list_of_stds, labels)):
        x_values = np.arange(nvals) + 1 + (i - nseries // 2) * offset  # Adjusted x-coordinates
        plt.errorbar(x_values, means, yerr=stds, fmt='.', capsize=3, label=label)

    if xticklabels is not None:
        plt.xticks(np.arange(nvals) + 1, xticklabels, rotation=45)
    else:
        plt.xticks([])
    ax.set_xlim(0.5, nvals + 0.5)

    if sec_xticklabels is not None:
        sec_nvals = len(sec_xticklabels)
        sec_xticks = nvals / sec_nvals * np.arange(sec_nvals) + (nvals / sec_nvals + 1) / 2
        sec = ax.secondary_xaxis(location=0)
        sec.set_xticks(sec_xticks, labels=sec_xticklabels)
        sec.tick_params('x', length=0)

        # lines between the classes:
        sec_xticks = nvals / sec_nvals * np.arange(sec_nvals + 1) + 0.5
        sec2 = ax.secondary_xaxis(location=0)
        sec2.set_xticks(sec_xticks, labels=[])
        sec2.tick_params('x', length=70, width=1)

    if xlabel is not None:
        plt.xlabel(xlabel)
    if ylabel is not None:
        plt.ylabel(ylabel)
    if drawtarget:
        plt.axhline(y=alpha, color='red', linestyle='--', linewidth=0.8, label=r'$\alpha$ (target)')
    if drawbounds:
        plt.axhspan(
            y_lower, y_upper,
            color='yellow', alpha=0.3, linewidth=0., label=r"Theoretical bounds for $\mathrm{\mathbb{E}}[L]$"
        )
    plt.legend(loc=loclegend)
    if logscale:
        plt.yscale('log')
    kwargs = {}
    if ymin is not None:
        kwargs.update(bottom=ymin)
    if ymax is not None:
        kwargs.update(top=ymax)
    if kwargs != {}:
        plt.ylim(**kwargs)

    plt.show()

In [ ]:
ylabel_mse = r"$\left\|\hat{\boldsymbol{\kappa}}_i - \boldsymbol{\kappa}_i\right\|_2^2$"

In [ ]:
means = [np.mean(mse_ks_test), np.mean(mse_wiener_test), np.mean(mse_mcalens_test)]
std_devs = [np.std(mse_ks_test), np.std(mse_wiener_test), np.std(mse_mcalens_test)]

plot_means_errs(
    means, std_devs,
    ylabel=ylabel_mse, drawtarget=False, drawbounds=False,
    figsize=(4, 3)
)

In [ ]:
methods = ["KS", "Wiener", "MCALens"]
for method, mean, std in zip(methods, means, std_devs):
    print(f"{method} --> MSE = {1e4*mean:.1f} +/- {1e4*std:.1f}")

Let us perform a statistical test to evaluate whether the MSE differences are statistically significant.

In [ ]:
_, p_value_ks_vs_wiener = stats.ttest_rel(
    mse_ks_test, mse_wiener_test
)
_, p_value_wiener_vs_mcalens = stats.ttest_rel(
    mse_wiener_test, mse_mcalens_test
)

print(f"KS vs Wiener --> p-value = {p_value_ks_vs_wiener:.1e}")
print(f"Wiener vs MCALens --> p-value = {p_value_wiener_vs_mcalens:.1e}")

### Miscoverage rates and prediction interval

In [ ]:
def get_metrics(pred, res, kappa, mask=None):

    pred_lo = pred - res
    pred_hi = pred + res

    # Error rate per image (over pixels)
    err = wlutils.miscoverage_rate(
        pred_lo, pred_hi, kappa, mask=mask
    )

    # Mean length of prediction intervals
    predinterv = wlutils.mean_predinterv(
        pred_lo, pred_hi, mask=mask
    )

    # Mean value for the lower and upper bounds
    lower = wlutils.mean_val(pred_lo, mask=mask)
    upper = wlutils.mean_val(pred_hi, mask=mask)

    return err, predinterv, lower, upper

In [ ]:
err_ks_test, predinterv_ks_test, lower_ks_test, upper_ks_test = get_metrics(
    kappa_ks_test, res_ks_test, kappa_test, mask=mask
)
err_wiener_test, predinterv_wiener_test, lower_wiener_test, upper_wiener_test = get_metrics(
    kappa_wiener_test, res_wiener_test, kappa_test, mask=mask
)
err_mcalens_test, predinterv_mcalens_test, lower_mcalens_test, upper_mcalens_test = get_metrics(
    kappa_mcalens_test, res_mcalens_test, kappa_test, mask=mask
)

In [ ]:
ylabel_err = (
    r"$L\left(\boldsymbol{\kappa}_i,\, \hat{\boldsymbol{\kappa}}_i^-,\, "
    r"\hat{\boldsymbol{\kappa}}_i^+\right)$"
)

In [ ]:
plot_means_errs(
    [np.mean(err_ks_test), np.mean(err_wiener_test), np.mean(err_mcalens_test)],
    [np.std(err_ks_test), np.std(err_wiener_test), np.std(err_mcalens_test)],
    ylabel=ylabel_err, drawbounds=False,
    ymin=0., figsize=(4, 3)
)

In [ ]:
confidence = 2. # Level of confidence (n-sigma)
alpha = wlutils.get_alpha_from_confidence(confidence)

min_nimgs_calib = wlutils.get_min_nimgs_calib(alpha)
print(f"Confidence level of {confidence}-sigma --> alpha = {alpha:.1%}")
print(f"Smallest possible size for the calibration set (CQR) = {min_nimgs_calib}")

## Conformalized quantile regression

In [ ]:
def conformalize_get_metrics(
        cqr, pred_test, res_test, kappa_test, pred_calib, res_calib, kappa_calib,
        mask=None
):
    """
    Parameters
    ----------
    cqr (wlmmuq.cqr.BaseCQR)
    pred_test, res_test, kappa_test (numpy.ndarray)
        Point estimate, residual and ground truth, from the test set.
        Shape = (nimgs_test, width, width)
    pred_calib, res_calib, kappa_calib (numpy.ndarray)
        Point estimate, residual and ground truth, from the calibration set.
        Shape = (nimgs_calib, width, width)
    mask (numpy.ndarray, default=None)
        Array of booleans, to exclude pixels outside the COSMOS boundaries, or
        with no measured galaxy. Shape = (nx, ny)
    
    """
    # Apply CQR-like calibration
    res_cqr_test, quantile_vals, _ = cqr.conformalize(
        res_test, pred_calib, res_calib, kappa_calib
    )

    # Discard quantile values outside the mask
    if mask is not None:
        quantile_vals = quantile_vals[mask]

    # Get metrics
    err_cqr_test, predinterv_cqr_test, lower_cqr_test, upper_cqr_test = get_metrics(
        pred_test, res_cqr_test, kappa_test, mask=mask
    )

    return res_cqr_test, err_cqr_test, predinterv_cqr_test, lower_cqr_test, upper_cqr_test

### Additive CQR

In [ ]:
addcqr = wlcqr.AddCQR(alpha)

In [ ]:
res_ks_addcqr_test, err_ks_addcqr_test, predinterv_ks_addcqr_test, \
        lower_ks_addcqr_test, upper_ks_addcqr_test = conformalize_get_metrics(
    addcqr, kappa_ks_test, res_ks_test, kappa_test, kappa_ks_calib, res_ks_calib, kappa_calib,
    mask=mask
)

res_wiener_addcqr_test, err_wiener_addcqr_test, predinterv_wiener_addcqr_test, \
        lower_wiener_addcqr_test, upper_wiener_addcqr_test = conformalize_get_metrics(
    addcqr, kappa_wiener_test, res_wiener_test, kappa_test, kappa_wiener_calib, res_wiener_calib, kappa_calib,
    mask=mask
)

res_mcalens_addcqr_test, err_mcalens_addcqr_test, predinterv_mcalens_addcqr_test, \
        lower_mcalens_addcqr_test, upper_mcalens_addcqr_test = conformalize_get_metrics(
    addcqr, kappa_mcalens_test, res_mcalens_test, kappa_test, kappa_mcalens_calib, res_mcalens_calib, kappa_calib,
    mask=mask
)

### Multiplicative CQR

In [ ]:
multcqr = wlcqr.MultCQR(alpha)

In [ ]:
_, err_ks_multcqr_test, predinterv_ks_multcqr_test, \
        lower_ks_multcqr_test, upper_ks_multcqr_test = conformalize_get_metrics(
    multcqr, kappa_ks_test, res_ks_test, kappa_test, kappa_ks_calib, res_ks_calib, kappa_calib,
    mask=mask
)

_, err_wiener_multcqr_test, predinterv_wiener_multcqr_test, \
        lower_wiener_multcqr_test, upper_wiener_multcqr_test = conformalize_get_metrics(
    multcqr, kappa_wiener_test, res_wiener_test, kappa_test, kappa_wiener_calib, res_wiener_calib, kappa_calib,
    mask=mask
)

_, err_mcalens_multcqr_test, predinterv_mcalens_multcqr_test, \
        lower_mcalens_multcqr_test, upper_mcalens_multcqr_test = conformalize_get_metrics(
    multcqr, kappa_mcalens_test, res_mcalens_test, kappa_test, kappa_mcalens_calib, res_mcalens_calib, kappa_calib,
    mask=mask
)

### Chi-squared CQR

In [ ]:
chisqcqr = wlcqr.ChisqCQR(alpha, mask=mask)

In [ ]:
err_ks_chisqcqr_test = []
err_wiener_chisqcqr_test = []
err_mcalens_chisqcqr_test = []

predinterv_ks_chisqcqr_test = []
predinterv_wiener_chisqcqr_test = []
predinterv_mcalens_chisqcqr_test = []

lower_ks_chisqcqr_test = []
lower_wiener_chisqcqr_test = []
lower_mcalens_chisqcqr_test = []

upper_ks_chisqcqr_test = []
upper_wiener_chisqcqr_test = []
upper_mcalens_chisqcqr_test = []

Test with several values of $a$ (scale factor)

In [ ]:
scalefacts = np.linspace(.004, .012, 5)
for a in scalefacts:
    chisqcqr.a = a

    _, err_ks_chisqcqr_test_0, predinterv_ks_chisqcqr_test_0, \
            lower_ks_chisqcqr_test_0, upper_ks_chisqcqr_test_0 = conformalize_get_metrics(
        chisqcqr, kappa_ks_test, res_ks_test, kappa_test, kappa_ks_calib, res_ks_calib, kappa_calib,
        mask=mask
    )
    err_ks_chisqcqr_test.append(err_ks_chisqcqr_test_0)
    predinterv_ks_chisqcqr_test.append(predinterv_ks_chisqcqr_test_0)
    lower_ks_chisqcqr_test.append(lower_ks_chisqcqr_test_0)
    upper_ks_chisqcqr_test.append(upper_ks_chisqcqr_test_0)

    _, err_wiener_chisqcqr_test_0, predinterv_wiener_chisqcqr_test_0, \
            lower_wiener_chisqcqr_test_0, upper_wiener_chisqcqr_test_0 = conformalize_get_metrics(
        chisqcqr, kappa_wiener_test, res_wiener_test, kappa_test, kappa_wiener_calib, res_wiener_calib, kappa_calib,
        mask=mask
    )
    err_wiener_chisqcqr_test.append(err_wiener_chisqcqr_test_0)
    predinterv_wiener_chisqcqr_test.append(predinterv_wiener_chisqcqr_test_0)
    lower_wiener_chisqcqr_test.append(lower_wiener_chisqcqr_test_0)
    upper_wiener_chisqcqr_test.append(upper_wiener_chisqcqr_test_0)

    _, err_mcalens_chisqcqr_test_0, predinterv_mcalens_chisqcqr_test_0, \
            lower_mcalens_chisqcqr_test_0, upper_mcalens_chisqcqr_test_0 = conformalize_get_metrics(
        chisqcqr, kappa_mcalens_test, res_mcalens_test, kappa_test, kappa_mcalens_calib, res_mcalens_calib, kappa_calib,
        mask=mask
    )
    err_mcalens_chisqcqr_test.append(err_mcalens_chisqcqr_test_0)
    predinterv_mcalens_chisqcqr_test.append(predinterv_mcalens_chisqcqr_test_0)
    lower_mcalens_chisqcqr_test.append(lower_mcalens_chisqcqr_test_0)
    upper_mcalens_chisqcqr_test.append(upper_mcalens_chisqcqr_test_0)

### Visual representations

In [ ]:
skyshow_pred_bounds(
    kappa_ks_test[idx], res_ks_addcqr_test[idx], kappa_test[idx]
)

In [ ]:
%xdel res_ks_addcqr_test

In [ ]:
skyshow_pred_bounds(
    kappa_wiener_test[idx], res_wiener_addcqr_test[idx], kappa_test[idx]
)

In [ ]:
%xdel res_wiener_addcqr_test

In [ ]:
skyshow_pred_bounds(
    kappa_mcalens_test[idx], res_mcalens_addcqr_test[idx], kappa_test[idx]
)

In [ ]:
%xdel res_mcalens_addcqr_test

### Plot graphs

In [ ]:
list_of_err_ks_test = [
    err_ks_addcqr_test, err_ks_multcqr_test, *err_ks_chisqcqr_test
]
list_of_err_wiener_test = [
    err_wiener_addcqr_test, err_wiener_multcqr_test, *err_wiener_chisqcqr_test
]
list_of_err_mcalens_test = [
    err_mcalens_addcqr_test, err_mcalens_multcqr_test, *err_mcalens_chisqcqr_test
]

list_of_predinterv_ks_test = [
    predinterv_ks_addcqr_test, predinterv_ks_multcqr_test, *predinterv_ks_chisqcqr_test
]
list_of_predinterv_wiener_test = [
    predinterv_wiener_addcqr_test, predinterv_wiener_multcqr_test,
    *predinterv_wiener_chisqcqr_test
]
list_of_predinterv_mcalens_test = [
    predinterv_mcalens_addcqr_test, predinterv_mcalens_multcqr_test,
    *predinterv_mcalens_chisqcqr_test
]

list_of_lower_ks_test = [
    lower_ks_addcqr_test, lower_ks_multcqr_test, *lower_ks_chisqcqr_test
]
list_of_lower_wiener_test = [
    lower_wiener_addcqr_test, lower_wiener_multcqr_test, *lower_wiener_chisqcqr_test
]
list_of_lower_mcalens_test = [
    lower_mcalens_addcqr_test, lower_mcalens_multcqr_test, *lower_mcalens_chisqcqr_test
]

list_of_upper_ks_test = [
    upper_ks_addcqr_test, upper_ks_multcqr_test, *upper_ks_chisqcqr_test
]
list_of_upper_wiener_test = [
    upper_wiener_addcqr_test, upper_wiener_multcqr_test, *upper_wiener_chisqcqr_test
]
list_of_upper_mcalens_test = [
    upper_mcalens_addcqr_test, upper_mcalens_multcqr_test, *upper_mcalens_chisqcqr_test
]

In [ ]:
xticklabels_cqr = [
    r"Additive", r"Multiplicative", r"$\chi^2$, $a =$.004", r"$\chi^2$, $a =$.006",
    r"$\chi^2$, $a =$.008", r"$\chi^2$, $a =$.010", r"$\chi^2$, $a =$.012"
]

Miscoverage rate

In [ ]:
# Calculate mean and standard deviation for each statistical series
means_err_ks_cqr = [np.mean(err) for err in list_of_err_ks_test]
stds_err_ks_cqr = [np.std(err) for err in list_of_err_ks_test]

means_err_wiener_cqr = [np.mean(err) for err in list_of_err_wiener_test]
stds_err_wiener_cqr = [np.std(err) for err in list_of_err_wiener_test]

means_err_mcalens_cqr = [np.mean(err) for err in list_of_err_mcalens_test]
stds_err_mcalens_cqr = [np.std(err) for err in list_of_err_mcalens_test]

In [ ]:
lower_bound_proba, upper_bound_proba = addcqr.get_bounds_proba(nimgs_calib)

In [ ]:
plot_means_errs(
    [means_err_ks_cqr, means_err_wiener_cqr, means_err_mcalens_cqr],
    [stds_err_ks_cqr, stds_err_wiener_cqr, stds_err_mcalens_cqr],
    xticklabels_cqr, ylabel=ylabel_err,
    y_lower=lower_bound_proba, y_upper=upper_bound_proba,
    ymin=0., ymax=0.055
)

Mean length of prediction intervals

In [ ]:
# Calculate mean and standard deviation for each statistical series
means_predinterv_ks_cqr = [np.mean(err) for err in list_of_predinterv_ks_test]
stds_predinterv_ks_cqr = [np.std(err) for err in list_of_predinterv_ks_test]

means_predinterv_wiener_cqr = [np.mean(err) for err in list_of_predinterv_wiener_test]
stds_predinterv_wiener_cqr = [np.std(err) for err in list_of_predinterv_wiener_test]

means_predinterv_mcalens_cqr = [np.mean(err) for err in list_of_predinterv_mcalens_test]
stds_predinterv_mcalens_cqr = [np.std(err) for err in list_of_predinterv_mcalens_test]

Lower and upper bounds of prediction intervals

In [ ]:
ylabel_predinterv = (
    r"$\left\langle \hat{\boldsymbol{\kappa}}_i^- \right\rangle,\, "
    r"\left\langle \hat{\boldsymbol{\kappa}}_i^+ \right\rangle$"
)

In [ ]:
def plot_confidence_bounds(
        list_of_lower, list_of_upper, xticklabels, sec_xticklabels=None,
        xlabel=None, ylabel=None, logscale=False, ymin=None, ymax=None, loclegend=None
):
    """
    Plot minimum and maximum values.
    
    """
    nseries = len(list_of_lower)
    assert len(list_of_upper) == nseries
    nvals = len(xticklabels)
    offset = 0.2  # Adjust the offset as needed
    labels = ["KS", "Wiener", "MCALens"]

    _, ax = plt.subplots(figsize=(6, 3))
    for i, (lower, upper, label) in enumerate(zip(list_of_lower, list_of_upper, labels)):
        x_values = np.arange(nvals) + 1 + (i - nseries // 2) * offset  # Adjusted x-coordinates
        lengths = [hi - lo for lo, hi in zip(lower, upper)]
        plt.bar(x_values, lengths, bottom=lower, width=0.1, label=label)

    plt.xticks(np.arange(nvals) + 1, xticklabels, rotation=45)
    ax.set_xlim(0.5, nvals + 0.5)

    if sec_xticklabels is not None:
        sec_nvals = len(sec_xticklabels)
        sec_xticks = nvals / sec_nvals * np.arange(sec_nvals) + (nvals / sec_nvals + 1) / 2
        sec = ax.secondary_xaxis(location=0)
        sec.set_xticks(sec_xticks, labels=sec_xticklabels)
        sec.tick_params('x', length=0)

        # lines between the classes:
        sec_xticks = nvals / sec_nvals * np.arange(sec_nvals + 1) + 0.5
        sec2 = ax.secondary_xaxis(location=0)
        sec2.set_xticks(sec_xticks, labels=[])
        sec2.tick_params('x', length=70, width=1)

    if xlabel is not None:
        plt.xlabel(xlabel)
    if ylabel is not None:
        plt.ylabel(ylabel)
    plt.legend(loc=loclegend)
    if logscale:
        plt.yscale('log')
    kwargs = {}
    if ymin is not None:
        kwargs.update(bottom=ymin)
    if ymax is not None:
        kwargs.update(top=ymax)
    if kwargs != {}:
        plt.ylim(**kwargs)

    plt.show()

In [ ]:
# Calculate mean and standard deviation for each statistical series
means_lower_ks_cqr = [np.mean(lower) for lower in list_of_lower_ks_test]
means_upper_ks_cqr = [np.mean(upper) for upper in list_of_upper_ks_test]

means_lower_wiener_cqr = [np.mean(lower) for lower in list_of_lower_wiener_test]
means_upper_wiener_cqr = [np.mean(upper) for upper in list_of_upper_wiener_test]

means_lower_mcalens_cqr = [np.mean(lower) for lower in list_of_lower_mcalens_test]
means_upper_mcalens_cqr = [np.mean(upper) for upper in list_of_upper_mcalens_test]

In [ ]:
plot_confidence_bounds(
    [means_lower_ks_cqr, means_lower_wiener_cqr, means_lower_mcalens_cqr],
    [means_upper_ks_cqr, means_upper_wiener_cqr, means_upper_mcalens_cqr],
    xticklabels_cqr, ylabel=ylabel_predinterv, ymin=-0.15, ymax=0.15,
    loclegend="lower right"
)

## Risk-controlling predictions sets

In [ ]:
def calibratercps_get_metrics(
        rcps, pred_test, res_test, kappa_test, pred_calib, res_calib, kappa_calib,
        mask=None
):
    """
    Parameters
    ----------
    rcps (wlmmuq.cqr.BaseRCPS)
    pred_test, res_test, kappa_test (numpy.ndarray)
        Point estimate, residual and ground truth, from the test set.
        Shape = (nimgs_test, width, width)
    pred_calib, res_calib, kappa_calib (numpy.ndarray)
        Point estimate, residual and ground truth, from the calibration set.
        Shape = (nimgs_calib, width, width)
    mask (numpy.ndarray, default=None)
        Array of booleans, to exclude pixels outside the COSMOS boundaries, or
        with no measured galaxy. Shape = (nx, ny)
    
    """
    # Apply RCPS-like calibration
    res_rcps_test, lamb = rcps.calibrate(
        res_test, pred_calib, res_calib, kappa_calib, mask=mask
    )

    # Get metrics
    err_rcps_test, predinterv_rcps_test, lower_rcps_test, upper_rcps_test = get_metrics(
        pred_test, res_rcps_test, kappa_test, mask=mask
    )

    return res_rcps_test, err_rcps_test, predinterv_rcps_test, lower_rcps_test, upper_rcps_test, lamb

In [ ]:
list_of_delta = [0.05, 0.2, 0.5]
xlabel = r"$\delta$"
xticklabels_rcps = 3 * ["Additive", "Multiplicative"]
sec_xticklabels = [
    "\n\n\n\n\n" + r"$\delta =$" + f"{delta:.1%}" for delta in list_of_delta
]

#### Additive RCPS

In [ ]:
addrcps = wlrcps.AddRCPS(alpha=alpha, delta=None, nimgs_calib=nimgs_calib)

In [ ]:
list_of_err_ks_addrcps_test = []
list_of_predinterv_ks_addrcps_test = []
list_of_lower_ks_addrcps_test = []
list_of_upper_ks_addrcps_test = []
list_of_lamb_ks_addrcps = []

list_of_err_wiener_addrcps_test = []
list_of_predinterv_wiener_addrcps_test = []
list_of_lower_wiener_addrcps_test = []
list_of_upper_wiener_addrcps_test = []
list_of_lamb_wiener_addrcps = []

list_of_err_mcalens_addrcps_test = []
list_of_predinterv_mcalens_addrcps_test = []
list_of_lower_mcalens_addrcps_test = []
list_of_upper_mcalens_addrcps_test = []
list_of_lamb_mcalens_addrcps = []

for delta in list_of_delta:
    print(f"delta = {delta:.1%}")
    addrcps.delta = delta

    print("\tCalibrate KS...")
    res, err, predinterv, lower, upper, lamb = calibratercps_get_metrics(
        addrcps, kappa_ks_test, res_ks_test, kappa_test,
        kappa_ks_calib, res_ks_calib, kappa_calib,
        mask=mask
    )
    list_of_err_ks_addrcps_test.append(err)
    list_of_predinterv_ks_addrcps_test.append(predinterv)
    list_of_lower_ks_addrcps_test.append(lower)
    list_of_upper_ks_addrcps_test.append(upper)
    list_of_lamb_ks_addrcps.append(lamb)
    skyshow_pred_bounds(
        kappa_ks_test[idx], res[idx], kappa_test[idx]
    )

    print("\tCalibrate Wiener...")
    res, err, predinterv, lower_wiener, upper_wiener, lamb = calibratercps_get_metrics(
        addrcps, kappa_wiener_test, res_wiener_test, kappa_test,
        kappa_wiener_calib, res_wiener_calib, kappa_calib,
        mask=mask
    )
    list_of_err_wiener_addrcps_test.append(err)
    list_of_predinterv_wiener_addrcps_test.append(predinterv)
    list_of_lower_wiener_addrcps_test.append(lower_wiener)
    list_of_upper_wiener_addrcps_test.append(upper_wiener)
    list_of_lamb_wiener_addrcps.append(lamb)
    skyshow_pred_bounds(
        kappa_wiener_test[idx], res[idx], kappa_test[idx]
    )

    print("\tCalibrate MCALens...")
    res, err, predinterv, lower_mcalens, upper_mcalens, lamb = calibratercps_get_metrics(
        addrcps, kappa_mcalens_test, res_mcalens_test, kappa_test,
        kappa_mcalens_calib, res_mcalens_calib, kappa_calib,
        mask=mask
    )
    list_of_err_mcalens_addrcps_test.append(err)
    list_of_predinterv_mcalens_addrcps_test.append(predinterv)
    list_of_lower_mcalens_addrcps_test.append(lower_mcalens)
    list_of_upper_mcalens_addrcps_test.append(upper_mcalens)
    list_of_lamb_mcalens_addrcps.append(lamb)
    skyshow_pred_bounds(
        kappa_mcalens_test[idx], res[idx], kappa_test[idx]
    )

In [ ]:
%xdel res

Plot calibration parameters $\lambda^{(\alpha,\, \delta)}$ for several velues of $\delta$

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(list_of_delta, list_of_lamb_ks_addrcps, label='KS')
plt.plot(list_of_delta, list_of_lamb_wiener_addrcps, label='Wiener')
plt.plot(list_of_delta, list_of_lamb_mcalens_addrcps, label='MCALens')
plt.xlabel(r"$\delta$")
plt.ylabel(r"$\lambda^{(\alpha,\, \delta)}$")
plt.ylim(bottom=0.)
plt.legend()
plt.show()

### Multiplicative RCPS

In [ ]:
multrcps = wlrcps.MultRCPS(alpha=alpha, delta=None, nimgs_calib=nimgs_calib)

In [ ]:
list_of_err_ks_multrcps_test = []
list_of_predinterv_ks_multrcps_test = []
list_of_lower_ks_multrcps_test = []
list_of_upper_ks_multrcps_test = []
list_of_lamb_ks_multrcps = []

list_of_err_wiener_multrcps_test = []
list_of_predinterv_wiener_multrcps_test = []
list_of_lower_wiener_multrcps_test = []
list_of_upper_wiener_multrcps_test = []
list_of_lamb_wiener_multrcps = []

list_of_err_mcalens_multrcps_test = []
list_of_predinterv_mcalens_multrcps_test = []
list_of_lower_mcalens_multrcps_test = []
list_of_upper_mcalens_multrcps_test = []
list_of_lamb_mcalens_multrcps = []

for delta in list_of_delta:
    print(f"delta = {delta:.1%}")
    multrcps.delta = delta

    print("\tCalibrate KS...")
    res, err, predinterv, lower, upper, lamb = calibratercps_get_metrics(
        multrcps, kappa_ks_test, res_ks_test, kappa_test,
        kappa_ks_calib, res_ks_calib, kappa_calib,
        mask=mask
    )
    list_of_err_ks_multrcps_test.append(err)
    list_of_predinterv_ks_multrcps_test.append(predinterv)
    list_of_lower_ks_multrcps_test.append(lower)
    list_of_upper_ks_multrcps_test.append(upper)
    list_of_lamb_ks_multrcps.append(lamb)
    skyshow_pred_bounds(
        kappa_ks_test[idx], res[idx], kappa_test[idx]
    )

    print("\tCalibrate Wiener...")
    res, err, predinterv, lower_wiener, upper_wiener, lamb = calibratercps_get_metrics(
        multrcps, kappa_wiener_test, res_wiener_test, kappa_test,
        kappa_wiener_calib, res_wiener_calib, kappa_calib,
        mask=mask
    )
    list_of_err_wiener_multrcps_test.append(err)
    list_of_predinterv_wiener_multrcps_test.append(predinterv)
    list_of_lower_wiener_multrcps_test.append(lower_wiener)
    list_of_upper_wiener_multrcps_test.append(upper_wiener)
    list_of_lamb_wiener_multrcps.append(lamb)
    skyshow_pred_bounds(
        kappa_wiener_test[idx], res[idx], kappa_test[idx]
    )

    print("\tCalibrate MCALens...")
    res, err, predinterv, lower_mcalens, upper_mcalens, lamb = calibratercps_get_metrics(
        multrcps, kappa_mcalens_test, res_mcalens_test, kappa_test,
        kappa_mcalens_calib, res_mcalens_calib, kappa_calib,
        mask=mask
    )
    list_of_err_mcalens_multrcps_test.append(err)
    list_of_predinterv_mcalens_multrcps_test.append(predinterv)
    list_of_lower_mcalens_multrcps_test.append(lower_mcalens)
    list_of_upper_mcalens_multrcps_test.append(upper_mcalens)
    list_of_lamb_mcalens_multrcps.append(lamb)
    skyshow_pred_bounds(
        kappa_mcalens_test[idx], res[idx], kappa_test[idx]
    )

Plot calibration parameters $\lambda^{(\alpha,\, \delta)}$ for several velues of $\delta$

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(list_of_delta, list_of_lamb_ks_multrcps, label='KS')
plt.plot(list_of_delta, list_of_lamb_wiener_multrcps, label='Wiener')
plt.plot(list_of_delta, list_of_lamb_mcalens_multrcps, label='MCALens')
plt.xlabel(r"$\delta$")
plt.ylabel(r"$\lambda^{(\alpha,\, \delta)}$")
plt.ylim(bottom=0.)
plt.legend()
plt.show()

In [ ]:
%xdel kappa_calib

%xdel kappa_ks_calib
%xdel kappa_wiener_calib
%xdel kappa_mcalens_calib

%xdel res_ks_calib
%xdel res_wiener_calib
%xdel res_mcalens_calib

### Plot graphs

Miscoverage rate

In [ ]:
# Calculate mean and standard deviation for each statistical series
means_err_ks_rcps = []
stds_err_ks_rcps = []
for err_add, err_mult in zip(
        list_of_err_ks_addrcps_test, list_of_err_ks_multrcps_test
):
    means_err_ks_rcps += [np.mean(err_add), np.mean(err_mult)]
    stds_err_ks_rcps += [np.std(err_add), np.std(err_mult)]

means_err_wiener_rcps = []
stds_err_wiener_rcps = []
for err_add, err_mult in zip(
        list_of_err_wiener_addrcps_test, list_of_err_wiener_multrcps_test
):
    means_err_wiener_rcps += [np.mean(err_add), np.mean(err_mult)]
    stds_err_wiener_rcps += [np.std(err_add), np.std(err_mult)]

means_err_mcalens_rcps = []
stds_err_mcalens_rcps = []
for err_add, err_mult in zip(
        list_of_err_mcalens_addrcps_test, list_of_err_mcalens_multrcps_test
):
    means_err_mcalens_rcps += [np.mean(err_add), np.mean(err_mult)]
    stds_err_mcalens_rcps += [np.std(err_add), np.std(err_mult)]

In [ ]:
plot_means_errs(
    [means_err_ks_rcps, means_err_wiener_rcps, means_err_mcalens_rcps],
    [stds_err_ks_rcps, stds_err_wiener_rcps, stds_err_mcalens_rcps],
    xticklabels_rcps, sec_xticklabels=sec_xticklabels,
    ylabel=ylabel_err, drawbounds=False,
    ymin=0., ymax=0.055
)

Mean length of prediction intervals

In [ ]:
# Calculate mean and standard deviation for each statistical series
means_predinterv_ks_rcps = []
stds_predinterv_ks_rcps = []
for predinterv_add, predinterv_mult in zip(
        list_of_predinterv_ks_addrcps_test, list_of_predinterv_ks_multrcps_test
):
    means_predinterv_ks_rcps += [np.mean(predinterv_add), np.mean(predinterv_mult)]
    stds_predinterv_ks_rcps += [np.std(predinterv_add), np.std(predinterv_mult)]

means_predinterv_wiener_rcps = []
stds_predinterv_wiener_rcps = []
for predinterv_add, predinterv_mult in zip(
        list_of_predinterv_wiener_addrcps_test, list_of_predinterv_wiener_multrcps_test
):
    means_predinterv_wiener_rcps += [np.mean(predinterv_add), np.mean(predinterv_mult)]
    stds_predinterv_wiener_rcps += [np.std(predinterv_add), np.std(predinterv_mult)]

means_predinterv_mcalens_rcps = []
stds_predinterv_mcalens_rcps = []
for predinterv_add, predinterv_mult in zip(
        list_of_predinterv_mcalens_addrcps_test, list_of_predinterv_mcalens_multrcps_test
):
    means_predinterv_mcalens_rcps += [np.mean(predinterv_add), np.mean(predinterv_mult)]
    stds_predinterv_mcalens_rcps += [np.std(predinterv_add), np.std(predinterv_mult)]

In [ ]:
means_lower_ks_rcps = []
means_upper_ks_rcps = []
for lower_add, lower_mult, upper_add, upper_mult in zip(
        list_of_lower_ks_addrcps_test, list_of_lower_ks_multrcps_test,
        list_of_upper_ks_addrcps_test, list_of_upper_ks_multrcps_test
):
    means_lower_ks_rcps += [np.mean(lower_add), np.mean(lower_mult)]
    means_upper_ks_rcps += [np.mean(upper_add), np.mean(upper_mult)]

means_lower_wiener_rcps = []
means_upper_wiener_rcps = []
for lower_add, lower_mult, upper_add, upper_mult in zip(
        list_of_lower_wiener_addrcps_test, list_of_lower_wiener_multrcps_test,
        list_of_upper_wiener_addrcps_test, list_of_upper_wiener_multrcps_test
):
    means_lower_wiener_rcps += [np.mean(lower_add), np.mean(lower_mult)]
    means_upper_wiener_rcps += [np.mean(upper_add), np.mean(upper_mult)]

means_lower_mcalens_rcps = []
means_upper_mcalens_rcps = []
for lower_add, lower_mult, upper_add, upper_mult in zip(
        list_of_lower_mcalens_addrcps_test, list_of_lower_mcalens_multrcps_test,
        list_of_upper_mcalens_addrcps_test, list_of_upper_mcalens_multrcps_test
):
    means_lower_mcalens_rcps += [np.mean(lower_add), np.mean(lower_mult)]
    means_upper_mcalens_rcps += [np.mean(upper_add), np.mean(upper_mult)]

In [ ]:
plot_confidence_bounds(
    [means_lower_ks_rcps, means_lower_wiener_rcps, means_lower_mcalens_rcps],
    [means_upper_ks_rcps, means_upper_wiener_rcps, means_upper_mcalens_rcps],
    xticklabels_rcps, sec_xticklabels=sec_xticklabels, ylabel=ylabel_predinterv,
    ymin=-0.15, ymax=0.15, loclegend="lower right"
)

## Focus on higher-density regions

In [ ]:
minval_kappa = 0.05 # threshold below which pixels are discarded

In [ ]:
n_highdensity_pixels = kappa_test[kappa_test >= minval_kappa].size
n_pixels = kappa_test.size
print(
    f"{n_highdensity_pixels} pixels above {minval_kappa:.1e} "
    f"({n_highdensity_pixels / n_pixels:.2%} of total)"
)

Create a binary mask of high-density pixels, for each input image

In [ ]:
mask_highdensity = (kappa_test >= minval_kappa) * mask

In [ ]:
plt.figure(figsize=(9, 3))

plt.subplot(121)
wlutils.skyshow(
    kappa_test[idx], vmin=vmin, vmax=saturation*vmax, extent=extent,
    boundaries=(ra, dec), printxylabels=True, printxticks=True, printyticks=True,
    title="Convergence map"
)
plt.subplot(122)
wlutils.skyshow(
    mask_highdensity[idx], vmin=vmin, vmax=saturation*vmax, extent=extent,
    boundaries=(ra, dec), printxylabels=True, printxticks=True, printyticks=True,
    title="Corresponding mask"
)
plt.show()

To get results by filtering on high-density regions, simply replace keyword argument `mask=mask` by `mask=mask_highdensity` when calling the functions `wlutils.normalized_mse` and `get_metrics`.